# Crypt Composition Figures

This notebook builds publication-style summaries from the saved graph crypt segmentations. Dataset and timepoint selection, marker harmonization, and optional marker exclusivity follow the same conventions as `plot_marker_neighborhood.ipynb`.


In [88]:
# If you edit paths or grouping, re-run this cell and everything below.
import os
import warnings
import glob
import hashlib
import json
from functools import lru_cache

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from organograph.graph.io import load_cell_graph
from organograph.graph.access import graph_get

from organograph.crypts.analysis_markers import (
    bin_marker_positivity,
    get_marker_counts_per_patch,
    assign_coexpression_category,
)
from organograph.crypts.filters import filter_crypts_by_markers
from organograph.io_utils.segmentation_io import (
    load_graph_crypt_segmentation,
    load_mesh_crypt_segmentation,
)
from organograph.mesh.OrganoidMesh import OrganoidMesh
from organograph.mesh.hks import compute_hks
from organograph.plotting.colors import resolve_category_colors



## Configuration

Edit dataset paths, selected timepoints, marker preprocessing, filters, and plotting targets here. `DATA_SPECS` is generated from these settings for the existing crypt-loading helpers.


In [ ]:
# -----------------------
# CONFIG
# -----------------------
PROJECT_ROOT = os.path.abspath(os.getcwd())
if os.path.basename(PROJECT_ROOT) in {"notebooks", "notebooks_dev", "legacy"}:
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)
DATA_ROOT    = os.path.join(PROJECT_ROOT, "..", "NicoleData")

NOTEBOOK_NAME = "plot_crypt_compositions"
EXPORT_DIR = os.path.join(PROJECT_ROOT, "exports", NOTEBOOK_NAME)
os.makedirs(EXPORT_DIR, exist_ok=True)


def export_figure(fig, filename):
    """Save a Matplotlib figure as a tightly cropped PDF in the notebook export folder."""
    safe_name = "".join(
        char if char.isalnum() or char in {"_", "-"} else "_"
        for char in str(filename)
    ).strip("_")
    path = os.path.join(EXPORT_DIR, f"{safe_name}.pdf")
    fig.savefig(path, format="pdf", bbox_inches="tight")
    print(f"Saved figure: {path}")
    return path

# -----------------------------------------------------------------------------
# Shared data/loading settings used by all retained figures and exports.
# -----------------------------------------------------------------------------
DATASETS = ["20250929"]
SEG_SUBDIR = "crypt_segmentations_graph"
GRAPH_SUBDIR = "graphs_preprocessed"
SELECTED_TIMEPOINTS = ["day3p5", "day4", "day4p5", "day4p5-more"]  # Exact segmentation timepoint folders; None uses all.
MAX_CRYPTS_PER_TIMEPOINT = None  # e.g. 50 for a quick smoke run; None uses all retained crypts.
RANDOM_SEED = 7


def normalize_dataset_names(datasets):
    if isinstance(datasets, str):
        return [datasets]
    return [str(dataset) for dataset in datasets]


def normalize_timepoint_names(timepoints):
    if timepoints is None:
        return None
    if isinstance(timepoints, str):
        return [timepoints]
    return [str(timepoint) for timepoint in timepoints]


def timepoint_to_float(tp):
    return float(str(tp).replace("day", "").replace("p", ".").replace("-more", ""))


def timepoint_label(tp):
    return f"day {timepoint_to_float(tp):g}"


DATA_SPECS = [
    {
        "dataset": dataset,
        "seg_subdir": SEG_SUBDIR,
        "graph_subdir": GRAPH_SUBDIR,
        "timepoints": None if SELECTED_TIMEPOINTS is None else [timepoint],
        "label": timepoint_label(timepoint) if timepoint is not None else dataset,
    }
    for dataset in normalize_dataset_names(DATASETS)
    for timepoint in (normalize_timepoint_names(SELECTED_TIMEPOINTS) or [None])
]

# -----------------------------------------------------------------------------
# Optional marker preprocessing applied after graph loading and before analysis.
# The raw graph marker fields are not modified; these settings define the
# effective marker matrix used by filters, figures, and exports below.
# -----------------------------------------------------------------------------
ENABLE_MARKER_HARMONIZATION = True
MARKER_HARMONIZATION_RULES = {
    "KI67": ["KI67", "Cyclin A", "Cyclin D"],
}
MARKER_HARMONIZATION_KEEP_UNMAPPED = True

ENABLE_MARKER_EXCLUSIVITY = True
EXCLUSIVITY_RULES = {
    "LGR5":     ["Chroma", "Mucin 2", "AldoB", "Glucagon", "Agr2", "Serotonin", "Lysozyme"],
    "Chroma":   ["Mucin 2", "Glucagon", "Serotonin", "Lysozyme"],
    "Mucin 2":  ["Chroma", "Glucagon", "Serotonin", "Lysozyme"],
    "AldoB":    ["Chroma", "Mucin 2", "Glucagon", "Agr2", "Serotonin", "Lysozyme"],
    "Glucagon": ["Serotonin"],
    "Agr2":     ["Chroma", "Mucin 2", "Glucagon", "Serotonin", "Lysozyme"],
    "Serotonin":[],
    "Lysozyme": ["Chroma", "Glucagon", "Serotonin"],
    "Cyclin D": ["LGR5", "Chroma", "Mucin 2", "AldoB", "Glucagon", "Agr2", "Serotonin", "Lysozyme"],
    "Cyclin A": ["LGR5", "Chroma", "Mucin 2", "AldoB", "Glucagon", "Agr2", "Serotonin", "Lysozyme"],
    "KI67":     ["LGR5", "Chroma", "Mucin 2", "AldoB", "Glucagon", "Agr2", "Serotonin", "Lysozyme"],
}
EXCLUSIVITY_INCLUDE_UNASSIGNED = True
EXCLUSIVITY_UNASSIGNED_LABEL = "unassigned"

MARKER_PANEL_TARGETS = ["LGR5", "Chroma", "Mucin 2", "AldoB", "Glucagon", "Agr2", "Serotonin", "Lysozyme", "KI67"]
CRYPT_COMPOSITION_MARKERS = list(MARKER_PANEL_TARGETS)
if ENABLE_MARKER_EXCLUSIVITY and EXCLUSIVITY_INCLUDE_UNASSIGNED:
    CRYPT_COMPOSITION_MARKERS.append(EXCLUSIVITY_UNASSIGNED_LABEL)

# Optional marker names used only for coexpression_cat if present in a given graph.
COEX_MARKERS = {
    "LGR5": "LGR5",
    "Sero": "Serotonin",
    "Lyso": "Lysozyme",
}

# Optional crypt filter settings.
# This version resolves marker indices separately inside each graph.
USE_CRYPT_FILTER = True

CRYPT_FILTER_KW = dict(
    pos_marker_names=['LGR5', 'Chroma', 'Agr2', 'Serotonin', 'Lysozyme'],      # e.g. ["LGR5"]
    neg_marker_names=["AldoB"],
    pos_min=1,
    neg_min=2,
    roi_frac=0.8,
    require_all_pos=False,
)
CRYPT_COMPOSITION_ROI_FRAC = CRYPT_FILTER_KW["roi_frac"]
BUDDING_INDEX_S_STAR = 0.75
BUDDING_INDEX_TRANSITION_INTERVAL = (-0.05, 0.05)
BUDDING_INDEX_CATEGORY_ORDER = ["combined", "bulged", "weakly budded", "fully budded"]

MIN_POSITIVE = 1
POS_THRESHOLD = 1


TARGET_MARKERS = CRYPT_COMPOSITION_MARKERS
DERIVED_MARKER_GROUPS = {}
NORMALIZED_DISTANCE_BIN_EDGES = np.linspace(0.0, 1.2, 10)
GRAPH_HOP_MAX = 10


def ordered_unique(values):
    seen = set()
    out = []
    for value in values:
        if value not in seen:
            seen.add(value)
            out.append(value)
    return out


SAMPLE_ORDER = ordered_unique(d["label"] for d in DATA_SPECS)


## Load Crypt-Level Table

The inspection table is one row per retained crypt. It loads each graph segmentation, resolves its graph, applies the optional marker-aware crypt filter, and records marker counts plus saved morphology descriptors.


In [90]:
from organograph.io_utils.cells_table import harmonize_markers

def discover_timepoints(seg_dir):
    if not os.path.isdir(seg_dir):
        return []
    tps = []
    for name in sorted(os.listdir(seg_dir)):
        p = os.path.join(seg_dir, name)
        if os.path.isdir(p) and glob.glob(os.path.join(p, "*.npz")):
            tps.append(name)
    return tps


def _seg_get(seg, *keys, default=None):
    for k in keys:
        if k in seg:
            return seg[k]
    return default


def _safe_array(x):
    if x is None:
        return None
    return np.asarray(x)


def _as_list_of_lists(x):
    if x is None:
        return []
    if isinstance(x, list):
        return [list(xx) for xx in x]
    if isinstance(x, np.ndarray) and x.dtype == object:
        return [list(xx) for xx in x.tolist()]
    return [list(xx) for xx in x]


def _infer_marker_names(G):
    """
    Read marker names from the graph itself.
    """
    for k in ["marker_names", "markers", "marker_cols"]:
        v = G.graph.get(k, None)
        if v is not None:
            return list(v)

    markers_bin = graph_get(G, "markers_bin")
    n_markers = markers_bin.shape[1]
    return [f"marker_{i}" for i in range(n_markers)]


def _infer_label_uid(G, fallback=None):
    return G.graph.get("label_uid", None) or fallback


@lru_cache(maxsize=None)
def load_graph_cached(graph_path):
    return load_cell_graph(graph_path)


def _as_bool_marker_matrix(markers):
    X = np.asarray(markers)
    if X.ndim != 2:
        raise ValueError(f"Expected a 2D marker matrix, got shape {X.shape}")
    return X > 0


def marker_preprocess_cache_tag():
    """Stable tag for result files that depend on effective marker preprocessing."""
    payload = {
        "harmonization_enabled": bool(ENABLE_MARKER_HARMONIZATION),
        "harmonization_rules": MARKER_HARMONIZATION_RULES if ENABLE_MARKER_HARMONIZATION else {},
        "harmonization_keep_unmapped": bool(MARKER_HARMONIZATION_KEEP_UNMAPPED),
        "exclusivity_enabled": bool(ENABLE_MARKER_EXCLUSIVITY),
        "exclusivity_rules": EXCLUSIVITY_RULES if ENABLE_MARKER_EXCLUSIVITY else {},
        "include_unassigned": bool(EXCLUSIVITY_INCLUDE_UNASSIGNED),
        "unassigned_label": EXCLUSIVITY_UNASSIGNED_LABEL,
    }
    text = json.dumps(payload, sort_keys=True)
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:10]


def marker_alias_candidates(marker_name):
    marker = str(marker_name)
    aliases = {
        "ChromA": ["ChromA", "Chroma", "Chromogranin A", "ChromograninA"],
        "Chroma": ["Chroma", "ChromA", "Chromogranin A", "ChromograninA"],
        "Chromogranin A": ["Chromogranin A", "ChromograninA", "ChromA", "Chroma"],
        "ChromograninA": ["ChromograninA", "Chromogranin A", "ChromA", "Chroma"],
    }
    out = list(aliases.get(marker, [marker]))
    if ENABLE_MARKER_HARMONIZATION:
        for harmonized, sources in MARKER_HARMONIZATION_RULES.items():
            if marker == harmonized and harmonized not in out:
                out.append(harmonized)
            if marker == harmonized:
                out.extend([s for s in sources if s not in out])
            if marker in sources and harmonized not in out:
                out.append(harmonized)
    deduped = []
    for alias in out:
        if alias not in deduped:
            deduped.append(alias)
    return deduped


def marker_name_to_index(marker_names):
    return {str(name).strip().lower(): i for i, name in enumerate(marker_names)}


def resolve_marker_indices(marker_names, requested):
    name_to_idx = marker_name_to_index(marker_names)
    idx = []
    for marker in requested:
        for alias in marker_alias_candidates(marker):
            key = str(alias).strip().lower()
            if key in name_to_idx and name_to_idx[key] not in idx:
                idx.append(name_to_idx[key])
                break
    return idx


def marker_consumed_by_harmonization(marker_name):
    if not ENABLE_MARKER_HARMONIZATION:
        return False
    marker_key = str(marker_name).strip().lower()
    for out_name, source_names in MARKER_HARMONIZATION_RULES.items():
        out_key = str(out_name).strip().lower()
        for source_name in source_names:
            if str(source_name).strip().lower() == marker_key and marker_key != out_key:
                return True
    return False


def harmonize_marker_matrix(markers_bin, marker_names):
    X = _as_bool_marker_matrix(markers_bin)
    names = list(marker_names)
    if not ENABLE_MARKER_HARMONIZATION or not MARKER_HARMONIZATION_RULES:
        return X.astype(float), names

    used_source_indices = set()
    harmonized_cols = []
    harmonized_names = []
    for out_name, source_names in MARKER_HARMONIZATION_RULES.items():
        idx = resolve_marker_indices(names, source_names)
        if not idx:
            continue
        used_source_indices.update(idx)
        harmonized_cols.append(np.any(X[:, idx], axis=1))
        harmonized_names.append(str(out_name))

    cols = []
    out_names = []
    if MARKER_HARMONIZATION_KEEP_UNMAPPED:
        for i, name in enumerate(names):
            if i in used_source_indices:
                continue
            cols.append(X[:, i])
            out_names.append(str(name))

    for name, col in zip(harmonized_names, harmonized_cols):
        key = str(name).strip().lower()
        existing = [i for i, n in enumerate(out_names) if str(n).strip().lower() == key]
        if existing:
            cols[existing[0]] = np.asarray(cols[existing[0]], dtype=bool) | np.asarray(col, dtype=bool)
        else:
            cols.append(np.asarray(col, dtype=bool))
            out_names.append(name)

    if not cols:
        return np.zeros((X.shape[0], 0), dtype=float), []
    return np.column_stack(cols).astype(float), out_names


def apply_marker_exclusivity(markers_bin, marker_names):
    X_raw = _as_bool_marker_matrix(markers_bin)
    X = X_raw.copy()
    names = list(marker_names)
    if not ENABLE_MARKER_EXCLUSIVITY:
        return X.astype(float), names

    for marker, forbidden_markers in EXCLUSIVITY_RULES.items():
        if marker_consumed_by_harmonization(marker):
            continue
        marker_idx = resolve_marker_indices(names, [marker])
        if not marker_idx:
            continue
        forbidden_idx = resolve_marker_indices(names, forbidden_markers)
        if forbidden_idx:
            suppress = np.any(X_raw[:, forbidden_idx], axis=1)
            X[:, marker_idx[0]] = X_raw[:, marker_idx[0]] & ~suppress

    label_sources = []
    label_names = []
    for marker in EXCLUSIVITY_RULES:
        if marker_consumed_by_harmonization(marker):
            continue
        idx = resolve_marker_indices(names, [marker])
        if not idx:
            continue
        label_sources.append((idx[0], marker))
        if marker not in label_names:
            label_names.append(marker)

    if EXCLUSIVITY_INCLUDE_UNASSIGNED and EXCLUSIVITY_UNASSIGNED_LABEL not in label_names:
        label_names.append(EXCLUSIVITY_UNASSIGNED_LABEL)

    label_to_idx = {label: i for i, label in enumerate(label_names)}
    out = np.zeros((X.shape[0], len(label_names)), dtype=bool)
    assigned = np.zeros(X.shape[0], dtype=bool)
    for source_idx, label in label_sources:
        take = (X[:, source_idx] > 0) & ~assigned
        if np.any(take):
            out[take, label_to_idx[label]] = True
            assigned[take] = True
    if EXCLUSIVITY_INCLUDE_UNASSIGNED:
        out[~assigned, label_to_idx[EXCLUSIVITY_UNASSIGNED_LABEL]] = True
    return out.astype(float), label_names


def effective_marker_matrix(markers_bin, marker_names):
    X, names = harmonize_marker_matrix(markers_bin, marker_names)
    X, names = apply_marker_exclusivity(X, names)
    return X.astype(float), list(names)


def effective_marker_data_for_graph(G):
    processed_marker_names = _infer_marker_names(G)
    processed_markers_bin = np.asarray(graph_get(G, "markers_bin"))
    return effective_marker_matrix(processed_markers_bin, processed_marker_names)


def effective_marker_counts_for_cells(G, cell_ids):
    X, marker_names = effective_marker_data_for_graph(G)
    idx = np.asarray(list(cell_ids), dtype=np.int64)
    if idx.size == 0:
        return np.zeros(len(marker_names), dtype=np.int64), 0, marker_names
    counts = np.sum(X[idx] > 0, axis=0).astype(np.int64)
    return counts, int(idx.size), marker_names


def build_dataset_paths(data_spec, data_root):
    dataset = data_spec["dataset"]

    seg_subdir = data_spec.get("seg_subdir", "crypt_segmentations_graph")
    graph_subdir = data_spec.get("graph_subdir", "graphs_preprocessed")

    seg_dir = os.path.join(data_root, dataset, seg_subdir)
    graphs_dir = os.path.join(data_root, dataset, graph_subdir)

    return seg_dir, graphs_dir


def resolve_graph_path(seg, graphs_dir):
    """
    Prefer the graph path stored in the segmentation if it exists.
    Otherwise try to rebuild it relative to the dataset-specific graphs_dir.
    """
    graph_path = seg.get("graph_path", None)

    if graph_path is not None and os.path.exists(graph_path):
        return graph_path

    if graph_path is None:
        return None

    # fallback: use basename inside graphs_dir
    base = os.path.basename(graph_path)
    candidate = os.path.join(graphs_dir, base)
    if os.path.exists(candidate):
        return candidate

    return graph_path


def resolve_mesh_path(seg, dataset_root=None):
    mesh_path = seg.get("mesh_path", None)
    if mesh_path is None:
        return None
    return mesh_path


def iter_selected_seg_paths(data_specs, data_root, verbose=True):
    """
    Iterate through all requested segmentation files.
    Yields dicts with:
      dataset, timepoint, label, seg_path, seg_dir, graphs_dir
    """
    for ds in data_specs:
        seg_dir, graphs_dir = build_dataset_paths(ds, data_root)

        tps = ds.get("timepoints", None)
        if tps is None:
            tps = discover_timepoints(seg_dir)

        for tp in tps:
            tp_dir = os.path.join(seg_dir, tp)
            seg_paths = sorted(glob.glob(os.path.join(tp_dir, "*.npz")))

            if verbose and len(seg_paths) == 0:
                print(f"[warn] no segmentation files found in: {tp_dir}")

            for seg_path in seg_paths:
                yield {
                    "dataset": ds["dataset"],
                    "timepoint": tp,
                    "label": ds.get("label", f'{ds["dataset"]}:{tp}'),
                    "seg_path": seg_path,
                    "seg_dir": seg_dir,
                    "graphs_dir": graphs_dir,
                }
                

def infer_common_markers(data_specs, data_root, verbose=True):
    """
    Intersect marker names across all selected graphs.
    Returns common markers in the order they first appear.
    """
    marker_sets = []
    ordered_first = None

    for item in iter_selected_seg_paths(data_specs, data_root, verbose=verbose):
        seg_path = item["seg_path"]
        graphs_dir = item["graphs_dir"]

        try:
            seg = load_graph_crypt_segmentation(seg_path)
            graph_path = resolve_graph_path(seg, graphs_dir=graphs_dir)
            G = load_graph_cached(graph_path)
            _X_eff, marker_names = effective_marker_data_for_graph(G)
        except Exception as e:
            if verbose:
                print(f"[skip common-marker scan] {seg_path}: {e}")
            continue

        marker_set = set(marker_names)
        marker_sets.append(marker_set)

        if ordered_first is None:
            ordered_first = list(marker_names)

    if len(marker_sets) == 0:
        return []

    common = set.intersection(*marker_sets)
    if ordered_first is None:
        return sorted(common)

    return [m for m in ordered_first if m in common]


def make_graph_aware_crypt_filter(
    pos_marker_names=None,
    neg_marker_names=None,
    pos_min=2,
    neg_min=2,
    roi_frac=0.65,
    require_all_pos=True,
):
    pos_marker_names = [] if pos_marker_names is None else list(pos_marker_names)
    neg_marker_names = [] if neg_marker_names is None else list(neg_marker_names)

    def _crypt_filter(G, crypts_graph, seg, marker_names):
        markers_eff, marker_names_eff = effective_marker_data_for_graph(G)
        pos_markers = resolve_marker_indices(marker_names_eff, pos_marker_names)
        neg_markers = resolve_marker_indices(marker_names_eff, neg_marker_names)

        crypt_list = list(crypts_graph) if crypts_graph is not None else []
        keep = np.zeros(len(crypt_list), dtype=bool)
        dist_bottom = np.asarray(seg["d_crypts_graph"], dtype=float) if roi_frac is not None else None

        for j, crypt_cells in enumerate(crypt_list):
            idx = np.asarray(list(crypt_cells), dtype=np.int64)
            if idx.size == 0:
                continue
            if roi_frac is not None:
                if dist_bottom.ndim == 1:
                    dvals = dist_bottom[idx]
                else:
                    dvals = dist_bottom[j, idx]
                idx = idx[np.isfinite(dvals) & (dvals <= float(roi_frac))]
                if idx.size == 0:
                    continue

            counts = np.sum(markers_eff[idx] > 0, axis=0)
            if pos_markers:
                ok_pos = [(counts[k] >= float(pos_min)) for k in pos_markers]
                if require_all_pos and not all(ok_pos):
                    continue
                if not require_all_pos and not any(ok_pos):
                    continue
            if any(counts[k] >= float(neg_min) for k in neg_markers):
                continue
            keep[j] = True
        return keep

    return _crypt_filter


def lighten_color(rgb_or_hex, amount=0.65):
    import matplotlib.colors as mcolors
    c = np.array(mcolors.to_rgb(rgb_or_hex))
    return tuple((1 - amount) * c + amount * np.ones(3))


def load_seg_cached(seg_path):
    return load_graph_crypt_segmentation(seg_path)


def get_roi_cells_for_crypt(crypt_cells, dist_bottom, crypt_number=None, roi_frac=None):
    """
    Restrict crypt cells to those with dist_bottom <= roi_frac.

    Supports:
      - dist_bottom shape (N_cells,)          : one global per-cell array
      - dist_bottom shape (K_crypts, N_cells) : one row per crypt

    Parameters
    ----------
    crypt_cells : list[int]
        Cell ids belonging to this crypt.
    dist_bottom : array-like
        Either shape (N_cells,) or (K_crypts, N_cells).
    crypt_number : int or None
        Required when dist_bottom has shape (K_crypts, N_cells).
        Should be the original crypt index in the segmentation.
    roi_frac : float or None
        If None, keep all crypt cells.
        Otherwise keep only cells with dist_bottom <= roi_frac.

    Returns
    -------
    roi_cells : np.ndarray of int
    """
    idx = np.asarray(list(crypt_cells), dtype=np.int64)

    if roi_frac is None:
        return idx

    db = np.asarray(dist_bottom, dtype=float)

    if db.ndim == 1:
        db_row = db

    elif db.ndim == 2:
        if crypt_number is None:
            raise ValueError(
                "crypt_number is required when dist_bottom has shape (K_crypts, N_cells)."
            )
        if crypt_number < 0 or crypt_number >= db.shape[0]:
            raise ValueError(
                f"crypt_number={crypt_number} out of range for dist_bottom with shape {db.shape}."
            )
        db_row = db[int(crypt_number)]

    else:
        raise ValueError(
            "dist_bottom must have shape (N_cells,) or (K_crypts, N_cells)."
        )

    if np.any(idx < 0) or np.any(idx >= len(db_row)):
        raise ValueError("crypt_cells contains indices out of range for dist_bottom.")

    dvals = db_row[idx]
    keep = np.isfinite(dvals) & (dvals <= float(roi_frac))
    return idx[keep]


def get_axis_window_cells_for_crypt(
    crypt_cells,
    dist_bottom,
    crypt_number=None,
    lower=None,
    upper=None,
    include_lower=False,
    include_upper=True,
):
    idx = np.asarray(list(crypt_cells), dtype=np.int64)
    db = np.asarray(dist_bottom, dtype=float)

    if db.ndim == 1:
        db_row = db
    elif db.ndim == 2:
        if crypt_number is None:
            raise ValueError("crypt_number is required when dist_bottom has shape (K_crypts, N_cells).")
        if crypt_number < 0 or crypt_number >= db.shape[0]:
            raise ValueError(f"crypt_number={crypt_number} out of range for dist_bottom with shape {db.shape}.")
        db_row = db[int(crypt_number)]
    else:
        raise ValueError("dist_bottom must have shape (N_cells,) or (K_crypts, N_cells).")

    if np.any(idx < 0) or np.any(idx >= len(db_row)):
        raise ValueError("crypt_cells contains indices out of range for dist_bottom.")

    dvals = db_row[idx]
    keep = np.isfinite(dvals)
    if lower is not None:
        keep &= dvals >= float(lower) if include_lower else dvals > float(lower)
    if upper is not None:
        keep &= dvals <= float(upper) if include_upper else dvals < float(upper)
    return idx[keep]


def compute_crypt_shape_metrics(seg, crypt_idx, fullness_power=1.0):
    """
    Compute scalar shape metrics for one crypt from segmentation data.

    Metrics
    -------
    crypt_length : float
        L

    crypt_cmax : float
        max circumference inside crypt (d_discretized <= 1)

    crypt_slenderness : float
        L / C_max

    crypt_fullness : float
        Weighted mean normalized position of circumference:
            mu = (∫ d * C(d)^p dd) / (∫ C(d)^p dd)

    crypt_circ_pos_var : float
        Weighted variance of circumference position:
            var = (∫ (d-mu)^2 * C(d)^p dd) / (∫ C(d)^p dd)

    crypt_circ_pos_spread : float
        sqrt(var)

    crypt_circ_pos_spread_norm : float
        2 * sqrt(var), roughly scaled to [0, 1]

    crypt_budding_index_sstar : float
        (C(s_star) - C(1)) / C_max, where s_star is BUDDING_INDEX_S_STAR.
    """
    L_crypts = _safe_array(_seg_get(seg, "L_crypts", default=None))
    circumference_crypts = _safe_array(_seg_get(seg, "circumference_crypts", default=None))
    d_discretized = _safe_array(_seg_get(seg, "d_discretized", default=None))

    out = {
        "crypt_length": np.nan,
        "crypt_cmax": np.nan,
        "crypt_slenderness": np.nan,
        "crypt_fullness": np.nan,
        "crypt_circ_pos_var": np.nan,
        "crypt_circ_pos_spread": np.nan,
        "crypt_circ_pos_spread_norm": np.nan,
        "crypt_budding_index_sstar": np.nan,
    }
    if L_crypts is None or circumference_crypts is None or d_discretized is None:
        return out

    if crypt_idx < 0 or crypt_idx >= len(L_crypts):
        return out

    try:
        L = float(L_crypts[crypt_idx])
        C = np.asarray(circumference_crypts[crypt_idx], dtype=float)
        d = np.asarray(d_discretized, dtype=float)
    except Exception:
        return out

    if C.ndim != 1 or d.ndim != 1 or len(C) != len(d):
        return out

    finite = np.isfinite(d) & np.isfinite(C)
    if not np.any(finite):
        return out

    d_all = d[finite]
    C_all = C[finite]
    all_order = np.argsort(d_all)
    d_all = d_all[all_order]
    C_all = C_all[all_order]

    keep = np.isfinite(d) & np.isfinite(C) & (d <= 1.0)
    if not np.any(keep):
        return out

    d_roi = d[keep]
    C_roi = C[keep]

    order = np.argsort(d_roi)
    d_roi = d_roi[order]
    C_roi = C_roi[order]

    C_max = float(np.nanmax(C_roi)) if len(C_roi) > 0 else np.nan
    slenderness = float(L / C_max) if np.isfinite(L) and C_max > 0 else np.nan

    if np.isfinite(C_max) and C_max > 0 and d_all[0] <= 1.0 <= d_all[-1]:
        C_boundary = float(np.interp(1.0, d_all, C_all))
        s_star = float(BUDDING_INDEX_S_STAR)
        if d_all[0] <= s_star <= d_all[-1]:
            C_star = float(np.interp(s_star, d_all, C_all))
            budding_index_sstar = float((C_star - C_boundary) / C_max)
        else:
            budding_index_sstar = np.nan
    else:
        budding_index_sstar = np.nan

    p = float(fullness_power)
    if p <= 0:
        raise ValueError("fullness_power must be > 0")

    W = C_roi ** p
    denom = np.trapezoid(W, d_roi)

    if denom > 0:
        mu = float(np.trapezoid(d_roi * W, d_roi) / denom)
        var = float(np.trapezoid(((d_roi - mu) ** 2) * W, d_roi) / denom)
        var = max(var, 0.0)
        spread = float(np.sqrt(var))
        spread_norm = float(2.0 * spread)
    else:
        mu = np.nan
        var = np.nan
        spread = np.nan
        spread_norm = np.nan

    out.update({
        "crypt_length": L,
        "crypt_cmax": C_max,
        "crypt_slenderness": slenderness,
        "crypt_fullness": mu,
        "crypt_circ_pos_var": var,
        "crypt_circ_pos_spread": spread,
        "crypt_circ_pos_spread_norm": spread_norm,
        "crypt_budding_index_sstar": budding_index_sstar,
    })
    return out


def assign_budding_index_category(value, interval=BUDDING_INDEX_TRANSITION_INTERVAL):
    if not np.isfinite(value):
        return "unknown"
    lo, hi = map(float, interval)
    if value < lo:
        return "bulged"
    if value > hi:
        return "fully budded"
    return "weakly budded"


### Inspect Stored Segmentation Variables

This quick check loads one graph segmentation from each `DATA_SPECS` entry and prints every stored variable with shape and dtype. Use it to confirm whether fields such as `curvature_gauss_graph` are available for each dataset/timepoint.


In [91]:
def _candidate_data_roots(preferred_data_root):
    roots = [preferred_data_root]
    cwd = os.getcwd()
    for candidate in [
        os.path.join(cwd, "..", "NicoleData"),
        os.path.join(cwd, "NicoleData"),
        os.path.join(cwd, "..", "..", "NicoleData"),
    ]:
        candidate = os.path.abspath(candidate)
        if candidate not in roots:
            roots.append(candidate)
    return roots


def _seg_dir_has_npz(seg_dir, timepoints):
    if timepoints is None:
        return bool(glob.glob(os.path.join(seg_dir, "*", "*.npz")))
    return any(glob.glob(os.path.join(seg_dir, tp, "*.npz")) for tp in timepoints)


def _resolve_existing_dataset_paths(data_spec, data_root):
    timepoints = data_spec.get("timepoints", None)
    for root in _candidate_data_roots(data_root):
        seg_dir, graphs_dir = build_dataset_paths(data_spec, root)
        if _seg_dir_has_npz(seg_dir, timepoints):
            return root, seg_dir, graphs_dir
    seg_dir, graphs_dir = build_dataset_paths(data_spec, data_root)
    return data_root, seg_dir, graphs_dir


def print_one_segmentation_variables_per_dataset(data_specs, data_root):
    for spec in data_specs:
        label = spec.get("label", spec.get("dataset", "dataset"))
        dataset = spec["dataset"]
        used_root, seg_dir, _graphs_dir = _resolve_existing_dataset_paths(spec, data_root)
        timepoints = spec.get("timepoints", None) or discover_timepoints(seg_dir)

        print("=" * 88)
        print(f"dataset={dataset} | label={label}")
        print(f"data_root={used_root}")
        print(f"seg_dir={seg_dir}")

        found = False
        for tp in timepoints:
            seg_paths = sorted(glob.glob(os.path.join(seg_dir, tp, "*.npz")))
            if not seg_paths:
                print(f"  {tp}: no .npz files found")
                continue

            seg_path = seg_paths[0]
            found = True
            print(f"  timepoint={tp}")
            print(f"  example={seg_path}")

            z = np.load(seg_path, allow_pickle=True)
            for key in z.files:
                value = z[key]
                shape = getattr(value, "shape", None)
                dtype = getattr(value, "dtype", type(value).__name__)
                extra = ""
                if np.asarray(value).shape == ():
                    try:
                        extra = f" value={value.item()}"
                    except Exception:
                        extra = ""
                print(f"    {key:<28} shape={str(shape):<16} dtype={dtype}{extra}")
            break

        if not found:
            print("  no segmentation files found for requested timepoints")


print_one_segmentation_variables_per_dataset(DATA_SPECS, DATA_ROOT)


dataset=20250929 | label=day 3.5
data_root=/home/fmoller/Projects/LearningOrganoids/OrganoGraph/../NicoleData
seg_dir=/home/fmoller/Projects/LearningOrganoids/OrganoGraph/../NicoleData/20250929/crypt_segmentations_graph
  timepoint=day3p5
  example=/home/fmoller/Projects/LearningOrganoids/OrganoGraph/../NicoleData/20250929/crypt_segmentations_graph/day3p5/day3p5_A01_1.npz
    label_uid                    shape=()               dtype=<U12 value=day3p5_A01_1
    graph_label_uid              shape=()               dtype=<U12 value=day3p5_A01_1
    mesh_seg_label_uid           shape=()               dtype=<U12 value=day3p5_A01_1
    parsed_label_uid             shape=()               dtype=<U12 value=day3p5_A01_1
    timepoint                    shape=()               dtype=<U6 value=day3p5
    mesh_seg_path                shape=()               dtype=<U124 value=/home/fmoller/Projects/LearningOrganoids/OrganoGraph/../NicoleData/20250929/crypt_segmentations_mesh/day3p5/day3p5_A01_1.npz
   

In [92]:
def build_inspection_table_multi(
    data_specs,
    data_root,
    verbose=True,
    min_positive=1,
    pos_threshold=1,
    coex_markers=None,
    crypt_filter=None,
):
    rows_out = []

    coex_markers = {} if coex_markers is None else dict(coex_markers)

    for item in iter_selected_seg_paths(data_specs, data_root, verbose=verbose):
        dataset = item["dataset"]
        tp = item["timepoint"]
        label = item["label"]
        seg_path = item["seg_path"]
        graphs_dir = item["graphs_dir"]

        # load segmentation
        try:
            seg = load_graph_crypt_segmentation(seg_path)
        except Exception as e:
            if verbose:
                print(f"[skip] could not load segmentation {seg_path}: {e}")
            continue

        # resolve graph path using dataset-specific graphs_dir
        graph_path = resolve_graph_path(seg, graphs_dir=graphs_dir)

        # load graph
        try:
            G = load_graph_cached(graph_path)
        except Exception as e:
            if verbose:
                print(f"[skip] could not load graph {graph_path}: {e}")
            continue

        markers_eff, marker_names = effective_marker_data_for_graph(G)
        name_to_idx = marker_name_to_index(marker_names)

        crypts_graph = _as_list_of_lists(
            _seg_get(seg, "crypts_graph", "crypts_ll", "graph_crypts_ll", default=[])
        )
        constrictions = _safe_array(_seg_get(seg, "crypt_constrictions", default=None))
        elongations = _safe_array(_seg_get(seg, "crypt_elongations", "crypt_elongationss", default=None))
        graph_patch_sizes = _safe_array(_seg_get(seg, "graph_patch_sizes", default=None))

        label_uid = _seg_get(seg, "label_uid", default=None)
        label_uid = _infer_label_uid(G, fallback=label_uid)

        n_crypts = len(crypts_graph)

        # run filter once
        if crypt_filter is None:
            keep_idx = np.arange(n_crypts, dtype=np.int64)
        else:
            keep = crypt_filter(G, crypts_graph, seg, marker_names)

            if isinstance(keep, np.ndarray) and keep.dtype == bool:
                if len(keep) != n_crypts:
                    raise ValueError(
                        f"crypt_filter returned boolean mask of length {len(keep)}, "
                        f"expected {n_crypts}"
                    )
                keep_idx = np.flatnonzero(keep)
            else:
                keep_idx = np.asarray(keep, dtype=np.int64)

        if keep_idx.ndim != 1:
            raise ValueError("keep_idx must be a 1D array of crypt indices")
        if len(keep_idx) > 0:
            if keep_idx.min() < 0 or keep_idx.max() >= n_crypts:
                raise ValueError("crypt_filter returned out-of-range crypt indices")

        crypts_graph_f = [crypts_graph[j] for j in keep_idx]
        constrictions_f = constrictions[keep_idx] if constrictions is not None else None
        elongations_f = elongations[keep_idx] if elongations is not None else None
        graph_patch_sizes_f = graph_patch_sizes[keep_idx] if graph_patch_sizes is not None else None
        orig_crypt_numbers_f = keep_idx

        # coexpression indices if present in this graph
        i_LGR5 = resolve_marker_indices(marker_names, [coex_markers.get("LGR5", "")])
        i_Sero = resolve_marker_indices(marker_names, [coex_markers.get("Sero", "")])
        i_Lyso = resolve_marker_indices(marker_names, [coex_markers.get("Lyso", "")])
        i_LGR5 = i_LGR5[0] if i_LGR5 else None
        i_Sero = i_Sero[0] if i_Sero else None
        i_Lyso = i_Lyso[0] if i_Lyso else None

        for local_idx, crypt_cells in enumerate(crypts_graph_f):
            crypt_cells = list(crypt_cells)
            if len(crypt_cells) == 0:
                continue

            orig_j = int(orig_crypt_numbers_f[local_idx])

            shape_metrics = compute_crypt_shape_metrics(seg, orig_j)

            crypt_idx = np.asarray(crypt_cells, dtype=np.int64)
            num_pos_cells = np.sum(markers_eff[crypt_idx] > 0, axis=0).astype(np.int64)
            num_cells = int(crypt_idx.size)

            if num_pos_cells.shape[0] != len(marker_names):
                raise ValueError(
                    f"Marker count mismatch for {seg_path}: "
                    f"counts have length {num_pos_cells.shape[0]}, "
                    f"marker_names has length {len(marker_names)}"
                )

            has_marker = (num_pos_cells >= min_positive).astype(np.int64)

            if None not in (i_LGR5, i_Sero, i_Lyso):
                has_eff = np.sum(markers_eff[crypt_idx] > 0, axis=0) >= int(pos_threshold)
                c0 = has_eff[i_LGR5] and has_eff[i_Sero] and (not has_eff[i_Lyso])
                c1 = has_eff[i_LGR5] and has_eff[i_Lyso] and (not has_eff[i_Sero])
                c2 = has_eff[i_LGR5] and (not has_eff[i_Lyso]) and (not has_eff[i_Sero])
                c3 = has_eff[i_LGR5] and has_eff[i_Lyso] and has_eff[i_Sero]
                c4 = (not has_eff[i_LGR5]) and has_eff[i_Sero] and (not has_eff[i_Lyso])
                c5 = (not has_eff[i_LGR5]) and has_eff[i_Lyso] and (not has_eff[i_Sero])
                c6 = (not has_eff[i_LGR5]) and (not has_eff[i_Lyso]) and (not has_eff[i_Sero])
                c7 = (not has_eff[i_LGR5]) and has_eff[i_Lyso] and has_eff[i_Sero]
                hits = np.flatnonzero(np.array([c0, c1, c2, c3, c4, c5, c6, c7], dtype=bool))
                coexpression_cat = int(hits[0]) if hits.size == 1 else np.nan
            else:
                coexpression_cat = np.nan

            constr = (
                float(constrictions_f[local_idx])
                if constrictions_f is not None and local_idx < len(constrictions_f)
                else np.nan
            )

            row = {
                "dataset": dataset,
                "timepoint": tp,
                "sample_label": label,
                "label_uid": label_uid,
                "crypt_number": orig_j,
                "crypt_number_filtered": local_idx,
                "graph_path": graph_path,
                "mesh_path": seg.get("mesh_path", None),
                "seg_path": seg_path if os.path.exists(seg_path) else None,
                "has_seg": bool(os.path.exists(seg_path)),
                "num_cells": int(num_cells),
                "constriction": constr,
                "elongation": (
                    float(elongations_f[local_idx])
                    if elongations_f is not None and local_idx < len(elongations_f)
                    else np.nan
                ),
                "graph_patch_size": (
                    int(graph_patch_sizes_f[local_idx])
                    if graph_patch_sizes_f is not None and local_idx < len(graph_patch_sizes_f)
                    else int(num_cells)
                ),
                "coexpression_cat": coexpression_cat,
                "is_budded": (constr > 0) if not np.isnan(constr) else np.nan,
                "crypt_shape_class": ("budded" if constr > 0 else "bulged") if not np.isnan(constr) else "unknown",
                "crypt_cells": crypt_cells,
                "graph_marker_names": tuple(marker_names),
                "crypt_length": shape_metrics["crypt_length"],
                "crypt_cmax": shape_metrics["crypt_cmax"],
                "crypt_slenderness": shape_metrics["crypt_slenderness"],
                "crypt_fullness": shape_metrics["crypt_fullness"],
                "crypt_circ_pos_var": shape_metrics["crypt_circ_pos_var"],
                "crypt_circ_pos_spread": shape_metrics["crypt_circ_pos_spread"],
                "crypt_circ_pos_spread_norm": shape_metrics["crypt_circ_pos_spread_norm"],
                "crypt_budding_index_sstar": shape_metrics["crypt_budding_index_sstar"],
                "budding_index_category": assign_budding_index_category(shape_metrics["crypt_budding_index_sstar"]),
            }
            for k, mk in enumerate(marker_names):
                row[f"{mk}_n_pos"] = int(num_pos_cells[k])
                row[f"{mk}_frac_pos"] = float(num_pos_cells[k] / num_cells) if num_cells > 0 else np.nan
                row[f"{mk}_has"] = int(has_marker[k])

            rows_out.append(row)

    df = pd.DataFrame(rows_out)

    if len(df) == 0:
        return df

    has_cols = [c for c in df.columns if c.endswith("_has")]
    if len(has_cols) > 0:
        df["n_markers_present"] = df[has_cols].fillna(0).sum(axis=1)

    return df


In [93]:
crypt_filter = None
if USE_CRYPT_FILTER:
    crypt_filter = make_graph_aware_crypt_filter(**CRYPT_FILTER_KW)


available_common_markers = infer_common_markers(DATA_SPECS, DATA_ROOT, verbose=True)
common_markers = list(CRYPT_COMPOSITION_MARKERS)
print("AVAILABLE COMMON MARKERS =", available_common_markers)
print("PLOTTED/EXPORTED MARKERS =", common_markers)

inspection_df = build_inspection_table_multi(
    data_specs=DATA_SPECS,
    data_root=DATA_ROOT,
    verbose=True,
    min_positive=MIN_POSITIVE,
    pos_threshold=POS_THRESHOLD,
    coex_markers=COEX_MARKERS,
    crypt_filter=crypt_filter,
)

if MAX_CRYPTS_PER_TIMEPOINT is not None and not inspection_df.empty:
    inspection_df = (
        inspection_df.groupby(["dataset", "timepoint"], group_keys=False, sort=False)
        .head(int(MAX_CRYPTS_PER_TIMEPOINT))
        .reset_index(drop=True)
    )

print("inspection_df shape:", inspection_df.shape)
inspection_df.head()

print("sample groups:", SAMPLE_ORDER)


AVAILABLE COMMON MARKERS = ['LGR5', 'Chroma', 'Mucin 2', 'AldoB', 'Glucagon', 'Agr2', 'Serotonin', 'Lysozyme', 'KI67', 'unassigned']
PLOTTED/EXPORTED MARKERS = ['LGR5', 'Chroma', 'Mucin 2', 'AldoB', 'Glucagon', 'Agr2', 'Serotonin', 'Lysozyme', 'KI67', 'unassigned']
inspection_df shape: (917, 59)
sample groups: ['day 3.5', 'day 4', 'day 4.5']


## Tunable Budding Index

The histogram shows the circumference-profile metric `(C(s*) - C(1)) / Cmax` at the selected `s*`, with the weakly budded interval highlighted.


In [ ]:
def plot_budding_index_category_histogram(
    inspection_df,
    metric_col="crypt_budding_index_sstar",
    category_col="budding_index_category",
    group_col="sample_label",
    group_order=SAMPLE_ORDER,
    bins=30,
    figsize=None,
    dpi=170,
):
    if metric_col not in inspection_df.columns:
        raise KeyError(f"Missing required column: {metric_col}")
    if category_col not in inspection_df.columns:
        raise KeyError(f"Missing required column: {category_col}")
    if group_col not in inspection_df.columns:
        raise KeyError(f"Missing required column: {group_col}")

    vals = inspection_df[metric_col].to_numpy(dtype=float)
    finite = np.isfinite(vals)
    vals_finite = vals[finite]
    cats = inspection_df.loc[finite, category_col].astype(str).to_numpy()
    if vals_finite.size == 0:
        raise ValueError(f"No finite values found for {metric_col}")

    lo = float(np.nanmin(vals_finite))
    hi = float(np.nanmax(vals_finite))
    pad = 0.05 * max(hi - lo, 1e-6)
    bin_edges = np.linspace(lo - pad, hi + pad, int(bins) + 1)

    group_order = [g for g in list(group_order) if g in set(inspection_df[group_col])]
    groups = [("all timepoints", inspection_df)]
    groups.extend((str(group), inspection_df[inspection_df[group_col] == group]) for group in group_order)

    category_colors = {
        "bulged": "#7FA6C7",
        "weakly budded": "#E5C45E",
        "fully budded": "#C96B5B",
    }

    if figsize is None:
        figsize = (3.1 * len(groups), 3.4)

    fig, axes = plt.subplots(1, len(groups), figsize=figsize, dpi=dpi, sharex=True, sharey=True)
    axes = np.atleast_1d(axes)

    lo_thr, hi_thr = map(float, BUDDING_INDEX_TRANSITION_INTERVAL)
    for ax, (group_label, group_df) in zip(axes, groups):
        group_vals = group_df[metric_col].to_numpy(dtype=float)
        group_finite = np.isfinite(group_vals)
        group_vals = group_vals[group_finite]
        group_cats = group_df.loc[group_finite, category_col].astype(str).to_numpy()
        hist_values = [group_vals[group_cats == category] for category in ["bulged", "weakly budded", "fully budded"]]
        ax.hist(
            hist_values,
            bins=bin_edges,
            stacked=True,
            color=[category_colors[category] for category in ["bulged", "weakly budded", "fully budded"]],
            edgecolor="white",
            linewidth=0.7,
            label=["bulged", "weakly budded", "fully budded"],
        )
        ax.axvspan(lo_thr, hi_thr, color=category_colors["weakly budded"], alpha=0.18, linewidth=0)
        ax.axvline(lo_thr, color="black", linewidth=0.8, linestyle=":")
        ax.axvline(hi_thr, color="black", linewidth=0.8, linestyle=":")
        ax.axvline(0.0, color="black", linewidth=0.8, linestyle="--", alpha=0.65)
        ax.set_title(f"{group_label}\nN={group_vals.size}", fontsize=8)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_ylabel("crypt count")
    fig.supxlabel(f"BI(s*={BUDDING_INDEX_S_STAR:g})", fontsize=10)
    fig.suptitle("Budding index categories", fontsize=10)
    handles = [plt.Rectangle((0, 0), 1, 1, color=category_colors[category]) for category in ["bulged", "weakly budded", "fully budded"]]
    fig.legend(handles, ["bulged", "weakly budded", "fully budded"], frameon=False, loc="upper right", fontsize=8)
    fig.tight_layout()
    return fig, axes


fig_bi_category_hist, ax_bi_category_hist = plot_budding_index_category_histogram(inspection_df)


## Pooled Marker Composition

These bars pool all cells in retained crypts within each display group and show the marker composition shared by all selected datasets.


In [ ]:
def summarize_markers_per_sample(inspection_df):
    rows = []
    for sample_label, g in inspection_df.groupby("sample_label", sort=False):
        marker_sets = g["graph_marker_names"].dropna().tolist()
        unique_markers = sorted(set().union(*[set(ms) for ms in marker_sets])) if len(marker_sets) else []
        rows.append({
            "sample_label": sample_label,
            "n_crypts": len(g),
            "markers_found": unique_markers,
        })
    return pd.DataFrame(rows)

marker_summary_df = summarize_markers_per_sample(inspection_df)
marker_summary_df

def build_pooled_common_marker_composition_table(
    inspection_df,
    common_markers,
    sample_col="sample_label",
    roi_frac=None,
):
    """
    Build one row per sample_label.

    For each common marker:
        pct(marker) = 100 * (total positive ROI cells for marker) / (total ROI cells)

    Then:
        pct(other) = 100 - sum_i pct(common_marker_i)

    Parameters
    ----------
    inspection_df : pd.DataFrame
        Must contain at least:
          - graph_path
          - seg_path
          - crypt_cells
          - crypt_number
          - sample_label

    common_markers : list[str]
        Markers that should be shown explicitly.

    roi_frac : float or None
        If not None, only count cells with seg["d_crypts_graph"] <= roi_frac.
        Should be between 0 and 1.

    Notes
    -----
    - Cells may contribute to multiple common markers.
    - Therefore 'other' is defined by subtraction exactly as requested.
    - AldoB is included in 'other' automatically unless it is in common_markers.
    """
    if roi_frac is not None:
        roi_frac = float(roi_frac)
        if not (0.0 <= roi_frac <= 1.0):
            raise ValueError("roi_frac must be in [0, 1] or None.")

    rows = []

    for sample_label, g in inspection_df.groupby(sample_col, sort=False):
        total_roi_cells = 0
        marker_pos_counts = {mk: 0 for mk in common_markers}
        n_crypts_used = 0

        for _, r in g.iterrows():
            graph_path = r["graph_path"]
            seg_path = r["seg_path"]
            crypt_cells = r["crypt_cells"]
            crypt_number = int(r["crypt_number"])

            if pd.isna(seg_path) or seg_path is None:
                continue
            if graph_path is None:
                continue

            G = load_graph_cached(graph_path)
            seg = load_seg_cached(seg_path)

            if "d_crypts_graph" not in seg:
                if roi_frac is not None:
                    raise KeyError(
                        f"seg['d_crypts_graph'] missing for seg_path={seg_path}, "
                        "but roi_frac was requested."
                    )
                dist_bottom = None
            else:
                dist_bottom = np.asarray(seg["d_crypts_graph"], dtype=float)

            markers_eff, marker_names = effective_marker_data_for_graph(G)

            if roi_frac is None:
                roi_cells = np.asarray(list(crypt_cells), dtype=np.int64)
            else:
                roi_cells = get_roi_cells_for_crypt(
                    crypt_cells=crypt_cells,
                    dist_bottom=dist_bottom,
                    crypt_number=crypt_number,
                    roi_frac=roi_frac,
                )

            if roi_cells.size == 0:
                continue

            n_crypts_used += 1
            total_roi_cells += int(roi_cells.size)

            X = np.asarray(markers_eff[roi_cells], dtype=float)

            for mk in common_markers:
                idx = resolve_marker_indices(marker_names, [mk])
                if not idx:
                    continue
                k = idx[0]
                marker_pos_counts[mk] += int(np.sum(X[:, k] > 0))

        row = {
            "sample_label": sample_label,
            "n_crypts": int(n_crypts_used),
            "total_cells": int(total_roi_cells),
            "roi_frac": roi_frac,
        }

        if total_roi_cells <= 0:
            for mk in common_markers:
                row[f"{mk}_pct_cells"] = np.nan
            row["other_pct_cells"] = np.nan
            rows.append(row)
            continue

        sum_common = 0.0
        for mk in common_markers:
            pct = 100.0 * float(marker_pos_counts[mk]) / float(total_roi_cells)
            row[f"{mk}_pct_cells"] = pct
            sum_common += pct

        row["other_pct_cells"] = 100.0 - sum_common
        rows.append(row)

    return pd.DataFrame(rows)

def plot_pooled_common_marker_composition(
    pooled_df,
    dataset_order=None,
    common_markers=None,
    other_label="other",
    color_scheme=None,
    figsize=None,
    dpi=170,
    percent_text_threshold=7.0,
    title="",
    export_filename="pooled_common_marker_composition",
):
    """
    One stacked bar per dataset/timepoint label.

    For each bar:
      - each common marker is plotted with its percentage of positive cells
      - 'other' = 100 - sum(common marker percentages)

    Styling is matched to the compact proportions of
    plot_lgr5_composition_from_inspection_table():
      - compact figure
      - narrow bars
      - smaller fonts
      - legend on the right, outside axes, single column
    """
    df = pooled_df.copy()

    if len(df) == 0:
        raise ValueError("pooled_df is empty")

    if common_markers is None:
        common_markers = [
            c[:-10] for c in df.columns
            if c.endswith("_pct_cells") and c != "other_pct_cells"
        ]

    categories = list(common_markers) + [other_label]

    if dataset_order is None:
        dataset_order = list(pd.unique(df["sample_label"]))

    df = df.set_index("sample_label").reindex(dataset_order)

    # --- compute compact figure width automatically ---
    n = len(dataset_order)

    if figsize is None:
        # these numbers are tuned to your current style
        base_w = 0.8          # minimal margin + legend room
        per_bar_w = 0.85      # horizontal space per bar
        fig_w = base_w + per_bar_w * n
        fig_h = 4.2 * 0.80

        figsize = (fig_w, fig_h)

    x = np.arange(len(dataset_order), dtype=float) * 0.75
    w = 0.40

    color_map = resolve_category_colors(
        categories=categories,
        color_scheme=color_scheme,
        other_label=other_label,
    )

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    # leave room on the right for legend
    fig.subplots_adjust(right=0.72, top=0.84)

    bottom = np.zeros(len(dataset_order), dtype=float)

    for cat in categories:
        col = f"{cat}_pct_cells" if cat != other_label else "other_pct_cells"
        vals = df[col].to_numpy(dtype=float)

        bars = ax.bar(
            x,
            vals,
            width=w,
            bottom=bottom,
            color=color_map[cat],
            edgecolor="white",
            linewidth=0.7,
            zorder=3,
            label=cat,
        )

        for rect, val, btm in zip(bars, vals, bottom):
            if not np.isfinite(val) or val < percent_text_threshold:
                continue

            ax.text(
                rect.get_x() + rect.get_width() / 2,
                btm + val / 2,
                f"{val:.0f}%",
                ha="center",
                va="center",
                fontsize=5,
                fontweight="bold",
                color="black",
                clip_on=True,
                zorder=4,
            )

        bottom = bottom + np.nan_to_num(vals, nan=0.0)

    # N labels above bars
    trans = ax.get_xaxis_transform()
    Ns = df["n_crypts"].fillna(0).to_numpy(dtype=int)
    for j in range(len(dataset_order)):
        ax.text(
            x[j], 1.01, f"N={Ns[j]}",
            transform=trans,
            ha="center",
            va="bottom",
            fontsize=5,
            fontweight="bold",
            clip_on=False,
        )

    # axis styling
    ax.set_ylim(0, 100)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.grid(True, axis="y", linestyle="--", linewidth=0.5, alpha=0.5, zorder=0)

    ax.set_xticks(x)
    ax.set_xticklabels(dataset_order, fontsize=7)
    ax.set_ylabel("% cells", fontsize=7)
    ax.tick_params(axis="y", labelsize=7)

    roi_vals = df["roi_frac"].dropna().unique() if "roi_frac" in df.columns else []
    if len(roi_vals) == 1:
        ax.set_title(f"{title} (ROI ≤ {roi_vals[0]:.2f})", fontsize=8)
    else:
        ax.set_title(title, fontsize=8)

    # legend on right, outside, single column
    handles = [
        plt.Rectangle((0, 0), 1, 1, facecolor=color_map[cat], edgecolor="none")
        for cat in categories
    ]
    labels = categories

    ax.legend(
        handles,
        labels,
        frameon=False,
        ncol=1,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        fontsize=7,
        borderaxespad=0.0,
        handletextpad=0.5,
        columnspacing=0.8,
    )

    export_figure(fig, export_filename)
    plt.show()

    return df.reset_index()


def budding_index_axis_regions(s_star=BUDDING_INDEX_S_STAR):
    s_star = float(s_star)
    return [
        {"axis_region": "s<=1", "lower": None, "upper": 1.0, "include_lower": True, "include_upper": True},
        {"axis_region": f"s<{s_star:g}", "lower": None, "upper": s_star, "include_lower": True, "include_upper": False},
        {"axis_region": f"{s_star:g}<s<=1", "lower": s_star, "upper": 1.0, "include_lower": False, "include_upper": True},
    ]


def _composition_row_for_records(records, common_markers, region, category_label):
    total_cells = 0
    marker_pos_counts = {mk: 0 for mk in common_markers}
    n_crypts_used = 0

    for _, rec in records.iterrows():
        graph_path = rec.get("graph_path")
        seg_path = rec.get("seg_path")
        if graph_path is None or pd.isna(graph_path) or seg_path is None or pd.isna(seg_path):
            continue

        G = load_graph_cached(graph_path)
        seg = load_seg_cached(seg_path)
        if "d_crypts_graph" not in seg:
            raise KeyError(f"seg['d_crypts_graph'] missing for seg_path={seg_path}")

        roi_cells = get_axis_window_cells_for_crypt(
            crypt_cells=rec["crypt_cells"],
            dist_bottom=np.asarray(seg["d_crypts_graph"], dtype=float),
            crypt_number=int(rec["crypt_number"]),
            lower=region["lower"],
            upper=region["upper"],
            include_lower=region["include_lower"],
            include_upper=region["include_upper"],
        )
        if roi_cells.size == 0:
            continue

        markers_eff, marker_names = effective_marker_data_for_graph(G)
        X = np.asarray(markers_eff[roi_cells], dtype=float)
        n_crypts_used += 1
        total_cells += int(roi_cells.size)

        for mk in common_markers:
            idx = resolve_marker_indices(marker_names, [mk])
            if not idx:
                continue
            marker_pos_counts[mk] += int(np.sum(X[:, idx[0]] > 0))

    row = {
        "axis_region": region["axis_region"],
        "budding_index_category": category_label,
        "n_crypts": int(n_crypts_used),
        "total_cells": int(total_cells),
        "s_star": float(BUDDING_INDEX_S_STAR),
    }

    if total_cells <= 0:
        for mk in common_markers:
            row[f"{mk}_pct_cells"] = np.nan
        row["other_pct_cells"] = np.nan
        return row

    sum_common = 0.0
    for mk in common_markers:
        pct = 100.0 * float(marker_pos_counts[mk]) / float(total_cells)
        row[f"{mk}_pct_cells"] = pct
        sum_common += pct
    row["other_pct_cells"] = 100.0 - sum_common
    return row


def build_budding_index_region_composition_table(
    inspection_df,
    common_markers,
    category_col="budding_index_category",
    category_order=BUDDING_INDEX_CATEGORY_ORDER,
):
    if category_col not in inspection_df.columns:
        raise KeyError(f"Missing required column: {category_col}")

    rows = []
    regions = budding_index_axis_regions()
    for region in regions:
        for category in category_order:
            if category == "combined":
                records = inspection_df
            else:
                records = inspection_df[inspection_df[category_col] == category]
            rows.append(_composition_row_for_records(records, common_markers, region, category))
    return pd.DataFrame(rows)


def plot_budding_index_region_composition(
    composition_df,
    common_markers=None,
    category_order=BUDDING_INDEX_CATEGORY_ORDER,
    other_label="other",
    color_scheme=None,
    figsize=(8.4, 3.4),
    dpi=170,
    percent_text_threshold=7.0,
    title_label="all timepoints",
    export_filename="pooled_common_marker_composition_bi_regions",
):
    df = composition_df.copy()
    if df.empty:
        raise ValueError("composition_df is empty")

    if common_markers is None:
        common_markers = [
            c[:-10] for c in df.columns
            if c.endswith("_pct_cells") and c != "other_pct_cells"
        ]
    categories = list(common_markers) + [other_label]
    color_map = resolve_category_colors(categories=categories, color_scheme=color_scheme, other_label=other_label)

    axis_regions = [r["axis_region"] for r in budding_index_axis_regions()]
    fig, axes = plt.subplots(1, len(axis_regions), figsize=figsize, dpi=dpi, sharey=True)
    axes = np.atleast_1d(axes)
    x = np.arange(len(category_order), dtype=float)
    w = 0.58

    for ax, axis_region in zip(axes, axis_regions):
        sub = (
            df[df["axis_region"] == axis_region]
            .set_index("budding_index_category")
            .reindex(category_order)
        )
        bottom = np.zeros(len(category_order), dtype=float)
        for cat in categories:
            col = f"{cat}_pct_cells" if cat != other_label else "other_pct_cells"
            vals = sub[col].to_numpy(dtype=float)
            bars = ax.bar(x, vals, width=w, bottom=bottom, color=color_map[cat], edgecolor="white", linewidth=0.7, label=cat)
            for rect, val, btm in zip(bars, vals, bottom):
                if not np.isfinite(val) or val < percent_text_threshold:
                    continue
                ax.text(rect.get_x() + rect.get_width() / 2, btm + val / 2, f"{val:.0f}%", ha="center", va="center", fontsize=4.8, fontweight="bold", color="black", clip_on=True)
            bottom = bottom + np.nan_to_num(vals, nan=0.0)

        trans = ax.get_xaxis_transform()
        Ns = sub["n_crypts"].fillna(0).to_numpy(dtype=int)
        for j, n_crypts in enumerate(Ns):
            ax.text(x[j], 1.01, f"N={n_crypts}", transform=trans, ha="center", va="bottom", fontsize=5, fontweight="bold", clip_on=False)

        ax.set_title(axis_region, fontsize=8)
        ax.set_xticks(x)
        ax.set_xticklabels(category_order, rotation=35, ha="right", fontsize=6.5)
        ax.set_ylim(0, 100)
        ax.set_yticks([0, 25, 50, 75, 100])
        ax.grid(True, axis="y", linestyle="--", linewidth=0.5, alpha=0.5, zorder=0)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_ylabel("% cells", fontsize=7)
    fig.suptitle(f"Crypt ROI cells by BI category: {title_label} (s*={BUDDING_INDEX_S_STAR:g})", fontsize=9)
    handles = [plt.Rectangle((0, 0), 1, 1, facecolor=color_map[cat], edgecolor="none") for cat in categories]
    fig.legend(handles, categories, frameon=False, ncol=1, loc="center left", bbox_to_anchor=(0.92, 0.5), fontsize=7)
    fig.subplots_adjust(right=0.84, top=0.82, bottom=0.24, wspace=0.18)
    export_figure(fig, export_filename)
    plt.show()
    return df

pooled_composition_df = build_pooled_common_marker_composition_table(
    inspection_df=inspection_df,
    common_markers=common_markers,
    sample_col="sample_label",
    roi_frac=None,
)


print("composition_df shape:", pooled_composition_df.shape)
pooled_composition_df.head()

plot_df = plot_pooled_common_marker_composition(
    pooled_df=pooled_composition_df,
    dataset_order=SAMPLE_ORDER,
    common_markers=common_markers,
    percent_text_threshold=3.0,
    title="All crypt cells",
    export_filename="pooled_common_marker_composition_all_cells",
)

def safe_label_for_filename(label):
    return "".join(char if char.isalnum() or char in {"_", "-"} else "_" for char in str(label)).strip("_")


pooled_composition_roi_tables = []
plot_roi_tables = {}

pooled_composition_roi_all_df = build_budding_index_region_composition_table(
    inspection_df=inspection_df,
    common_markers=common_markers,
)
pooled_composition_roi_all_df["figure_group"] = "all timepoints"
pooled_composition_roi_tables.append(pooled_composition_roi_all_df)

plot_roi_tables["all timepoints"] = plot_budding_index_region_composition(
    composition_df=pooled_composition_roi_all_df,
    common_markers=common_markers,
    percent_text_threshold=3.0,
    title_label="all timepoints",
    export_filename="pooled_common_marker_composition_bi_regions_all_timepoints",
)

for sample_label in SAMPLE_ORDER:
    inspection_subset = inspection_df[inspection_df["sample_label"] == sample_label]
    if inspection_subset.empty:
        continue
    group_df = build_budding_index_region_composition_table(
        inspection_df=inspection_subset,
        common_markers=common_markers,
    )
    group_df["figure_group"] = sample_label
    pooled_composition_roi_tables.append(group_df)
    plot_roi_tables[sample_label] = plot_budding_index_region_composition(
        composition_df=group_df,
        common_markers=common_markers,
        percent_text_threshold=3.0,
        title_label=sample_label,
        export_filename=f"pooled_common_marker_composition_bi_regions_{safe_label_for_filename(sample_label)}",
    )

pooled_composition_roi_df = pd.concat(pooled_composition_roi_tables, ignore_index=True)
plot_roi_df = pooled_composition_roi_df.copy()

print("BI-region roi composition_df shape:", pooled_composition_roi_df.shape)
pooled_composition_roi_df.head()

plot_df, plot_roi_df


## LGR5 and Coexpression Composition

These summaries operate at crypt level: first by LGR5 status, then by the LGR5/Serotonin/Lysozyme coexpression categories.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# from colors import DEFAULT_MARKER_COLORS
DEFAULT_MARKER_COLORS = {
    "Agr2": "#359BD5",
    "Lysozyme": "#2C75D2",
    "Mucin 2": "#3EC1D9",
    "Chroma": "#D852CB",
    "Glucagon": "#983EE2",
    "Serotonin": "#F392E3",
    "LGR5": "#FFB431",
    "AldoB": "#F16C6A",
    "Cyclin D": "#808080",
    "Cyclin A": "#808080",
    "KI67": "#808080",
    "other": "#EBEBEB",
}


def build_lgr5_composition_from_inspection_table(
    inspection_df,
    sample_col="sample_label",
    lgr5_marker="LGR5",
    sero_marker="Serotonin",
):
    """
    Build a pooled table for plotting LGR5+ / LGR5- crypt composition
    and serotonin-positive fractions within each LGR5 category.
    """
    df = inspection_df.copy()

    lgr5_col = f"{lgr5_marker}_has"
    sero_col = f"{sero_marker}_has"

    if lgr5_col not in df.columns:
        raise KeyError(f"Missing required column: {lgr5_col}")
    if sero_col not in df.columns:
        raise KeyError(f"Missing required column: {sero_col}")

    rows = []

    for sample_label, g in df.groupby(sample_col, sort=False):
        valid = g[g[lgr5_col].isin([0, 1])].copy()
        N = len(valid)

        if N == 0:
            rows.append({
                "sample_label": sample_label,
                "N": 0,
                "lgr5_pos_frac": np.nan,
                "lgr5_neg_frac": np.nan,
                "sero_in_lgr5_pos_frac": np.nan,
                "sero_in_lgr5_neg_frac": np.nan,
            })
            continue

        pos_mask = valid[lgr5_col] == 1
        neg_mask = valid[lgr5_col] == 0

        n_pos = int(pos_mask.sum())
        n_neg = int(neg_mask.sum())

        lgr5_pos_frac = n_pos / N
        lgr5_neg_frac = n_neg / N

        sero_in_lgr5_pos_frac = (
            (valid.loc[pos_mask, sero_col] == 1).mean() if n_pos > 0 else np.nan
        )
        sero_in_lgr5_neg_frac = (
            (valid.loc[neg_mask, sero_col] == 1).mean() if n_neg > 0 else np.nan
        )

        rows.append({
            "sample_label": sample_label,
            "N": N,
            "lgr5_pos_frac": lgr5_pos_frac,
            "lgr5_neg_frac": lgr5_neg_frac,
            "sero_in_lgr5_pos_frac": sero_in_lgr5_pos_frac,
            "sero_in_lgr5_neg_frac": sero_in_lgr5_neg_frac,
        })

    return pd.DataFrame(rows)


def plot_lgr5_composition_from_inspection_table(
    inspection_df,
    dataset_order=None,
    sample_col="sample_label",
    lgr5_marker="LGR5",
    sero_marker="Serotonin",
    figsize=None,
    dpi=170,
    main_text_threshold=7.0,
    sero_inside_height_threshold=3.0,
):
    """
    Plot pooled LGR5+ / LGR5- crypt composition directly from inspection_df.

    Outer stacked bars:
      - LGR5+
      - LGR5-

    Inside each segment:
      - a serotonin sub-bar at 90% of full bar width
      - serotonin % always written
      - main LGR5 % placed in the remaining non-serotonin space
    """
    plot_df = build_lgr5_composition_from_inspection_table(
        inspection_df=inspection_df,
        sample_col=sample_col,
        lgr5_marker=lgr5_marker,
        sero_marker=sero_marker,
    )

    if dataset_order is None:
        dataset_order = list(pd.unique(inspection_df[sample_col]))

    pooled = plot_df.set_index("sample_label").reindex(dataset_order)

    n = len(dataset_order)
    x = np.arange(n, dtype=float) * 0.78
    w = 0.30
    sero_w = 0.80 * w

    if figsize is None:
        base_w = 1.2
        per_bar_w = 0.75
        fig_w = base_w + per_bar_w * n
        fig_h = 4.2 * 0.82
        figsize = (fig_w, fig_h)

    lgr5_pos_color = DEFAULT_MARKER_COLORS["LGR5"]
    lgr5_neg_color = "#000000"
    sero_color = DEFAULT_MARKER_COLORS["Serotonin"]

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    fig.subplots_adjust(top=0.84)

    outer_stack = np.column_stack([
        pooled["lgr5_pos_frac"].to_numpy(dtype=float),
        pooled["lgr5_neg_frac"].to_numpy(dtype=float),
    ])

    sero_within = np.column_stack([
        pooled["sero_in_lgr5_pos_frac"].to_numpy(dtype=float),
        pooled["sero_in_lgr5_neg_frac"].to_numpy(dtype=float),
    ])

    outer_colors = [lgr5_pos_color, lgr5_neg_color]
    bottom = np.zeros(n, dtype=float)

    for si in range(2):
        seg_vals = 100.0 * outer_stack[:, si]
        sero_frac = sero_within[:, si]
        sero_vals = seg_vals * sero_frac

        # Parent LGR5 segment
        bars = ax.bar(
            x,
            seg_vals,
            width=w,
            bottom=bottom,
            color=outer_colors[si],
            edgecolor="white",
            linewidth=0.8,
            zorder=2,
        )

        # Serotonin sub-bar, visually nested inside parent
        sero_bars = ax.bar(
            x,
            np.nan_to_num(sero_vals, nan=0.0),
            width=sero_w,
            bottom=bottom,
            color=sero_color,
            edgecolor="white",
            linewidth=0.4,
            zorder=3,
        )

        for rect, seg_v, sero_rect, sero_v_abs, sero_v_rel, btm in zip(
            bars, seg_vals, sero_bars, sero_vals, sero_frac, bottom
        ):
            if not np.isfinite(seg_v) or seg_v <= 0:
                continue

            xc = rect.get_x() + rect.get_width() / 2
            sero_h = np.nan_to_num(sero_v_abs, nan=0.0)
            main_text_color = "black" if si == 0 else "white"

            # Always show serotonin percentage if it is defined
            if np.isfinite(sero_v_rel):
                sero_label = f"{100 * sero_v_rel:.0f}%"

                if sero_h >= sero_inside_height_threshold:
                    # write inside pink bar
                    ax.text(
                        xc,
                        btm + sero_h / 2,
                        sero_label,
                        ha="center",
                        va="center",
                        fontsize=4.8,
                        fontweight="bold",
                        color="black",
                        zorder=4,
                        clip_on=True,
                    )
                else:
                    # very short bar: place just above pink sub-bar
                    y_text = min(btm + sero_h + 1.2, btm + seg_v - 0.8)
                    ax.text(
                        xc,
                        y_text,
                        sero_label,
                        ha="center",
                        va="bottom",
                        fontsize=4.8,
                        fontweight="bold",
                        color="black",
                        zorder=4,
                        clip_on=True,
                    )

            # Main LGR5 percentage in remaining non-serotonin region
            non_sero_v = seg_v - sero_h

            if non_sero_v >= main_text_threshold:
                y_main = btm + sero_h + non_sero_v / 2
                ax.text(
                    xc,
                    y_main,
                    f"{seg_v:.0f}%",
                    ha="center",
                    va="center",
                    fontsize=5,
                    fontweight="bold",
                    color=main_text_color,
                    zorder=4,
                    clip_on=True,
                )
            elif seg_v >= main_text_threshold:
                # fallback near top of segment
                y_main = btm + seg_v - min(1.4, 0.15 * seg_v)
                ax.text(
                    xc,
                    y_main,
                    f"{seg_v:.0f}%",
                    ha="center",
                    va="top",
                    fontsize=5,
                    fontweight="bold",
                    color=main_text_color,
                    zorder=4,
                    clip_on=True,
                )

        bottom += np.nan_to_num(seg_vals, nan=0.0)

    # N labels above bars
    trans = ax.get_xaxis_transform()
    Ns = pooled["N"].fillna(0).to_numpy(dtype=int)
    for j in range(n):
        ax.text(
            x[j],
            1.01,
            f"N={Ns[j]}",
            transform=trans,
            ha="center",
            va="bottom",
            fontsize=5,
            fontweight="bold",
            clip_on=False,
        )

    ax.set_ylim(0, 100)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.grid(True, axis="y", linestyle="--", linewidth=0.5, alpha=0.5, zorder=0)
    ax.set_xticks(x)
    ax.set_xticklabels(dataset_order, fontsize=7)
    ax.set_ylabel("% crypts", fontsize=7)
    ax.tick_params(axis="y", labelsize=7)

    handles = [
        plt.Rectangle((0, 0), 1, 1, facecolor=lgr5_pos_color, edgecolor="none"),
        plt.Rectangle((0, 0), 1, 1, facecolor=lgr5_neg_color, edgecolor="none"),
        plt.Rectangle((0, 0), 1, 1, facecolor=sero_color, edgecolor="none"),
    ]
    labels = ["LGR5+", "LGR5-", "Serotonin+ within LGR5 class"]

    fig.legend(
        handles,
        labels,
        frameon=False,
        ncol=3,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.02),
        fontsize=7,
        columnspacing=1.4,
        handletextpad=0.6,
    )

    export_figure(fig, "lgr5_crypt_composition")
    plt.show()

    return plot_df


def build_lgr5_by_budding_index_category_table(
    inspection_df,
    group_col="sample_label",
    group_order=SAMPLE_ORDER,
    category_col="budding_index_category",
    category_order=BUDDING_INDEX_CATEGORY_ORDER,
    lgr5_marker="LGR5",
):
    lgr5_col = f"{lgr5_marker}_has"
    for col in [group_col, category_col, lgr5_col]:
        if col not in inspection_df.columns:
            raise KeyError(f"Missing required column: {col}")

    groups = [("all timepoints", inspection_df)]
    groups.extend(
        (str(group), inspection_df[inspection_df[group_col] == group])
        for group in group_order
        if group in set(inspection_df[group_col])
    )

    rows = []
    for group_label, group_df in groups:
        for category in category_order:
            if category == "combined":
                valid = group_df[group_df[lgr5_col].isin([0, 1])]
            else:
                valid = group_df[(group_df[category_col] == category) & group_df[lgr5_col].isin([0, 1])]
            N = int(len(valid))
            n_lgr5_pos = int((valid[lgr5_col] == 1).sum()) if N > 0 else 0
            frac = float(n_lgr5_pos / N) if N > 0 else np.nan
            rows.append({
                "figure_group": group_label,
                "budding_index_category": category,
                "N": N,
                "n_lgr5_pos": n_lgr5_pos,
                "lgr5_pos_frac": frac,
                "lgr5_pos_pct": 100.0 * frac if np.isfinite(frac) else np.nan,
            })
    return pd.DataFrame(rows)


def plot_lgr5_by_budding_index_category(
    lgr5_category_df,
    group_order=None,
    category_order=BUDDING_INDEX_CATEGORY_ORDER,
    figsize=None,
    dpi=170,
    export_filename="lgr5_crypt_composition_by_bi_category",
):
    df = lgr5_category_df.copy()
    if df.empty:
        raise ValueError("lgr5_category_df is empty")

    if group_order is None:
        group_order = list(pd.unique(df["figure_group"]))

    if figsize is None:
        figsize = (3.1 * len(group_order), 3.3)

    category_colors = {
        "combined": "#BDBDBD",
        "bulged": "#7FA6C7",
        "weakly budded": "#E5C45E",
        "fully budded": "#C96B5B",
    }

    fig, axes = plt.subplots(1, len(group_order), figsize=figsize, dpi=dpi, sharey=True)
    axes = np.atleast_1d(axes)
    x = np.arange(len(category_order), dtype=float)

    for ax, group_label in zip(axes, group_order):
        sub = (
            df[df["figure_group"] == group_label]
            .set_index("budding_index_category")
            .reindex(category_order)
        )
        vals = sub["lgr5_pos_pct"].to_numpy(dtype=float)
        colors = [category_colors.get(cat, "#BDBDBD") for cat in category_order]
        bars = ax.bar(x, vals, width=0.58, color=colors, edgecolor="white", linewidth=0.7)
        Ns = sub["N"].fillna(0).to_numpy(dtype=int)
        for rect, val, N in zip(bars, vals, Ns):
            if np.isfinite(val):
                ax.text(rect.get_x() + rect.get_width() / 2, min(val + 2.0, 98.0), f"{val:.0f}%", ha="center", va="bottom", fontsize=5.5, fontweight="bold")
            ax.text(rect.get_x() + rect.get_width() / 2, 1.01, f"N={N}", transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=5, fontweight="bold", clip_on=False)

        ax.set_title(str(group_label), fontsize=8)
        ax.set_xticks(x)
        ax.set_xticklabels(category_order, rotation=35, ha="right", fontsize=6.5)
        ax.set_ylim(0, 100)
        ax.set_yticks([0, 25, 50, 75, 100])
        ax.grid(True, axis="y", linestyle="--", linewidth=0.5, alpha=0.5, zorder=0)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_ylabel("% LGR5+ crypts", fontsize=7)
    fig.suptitle(f"LGR5+ crypts by BI category (s*={BUDDING_INDEX_S_STAR:g})", fontsize=9)
    fig.subplots_adjust(top=0.80, bottom=0.28, wspace=0.18)
    export_figure(fig, export_filename)
    plt.show()
    return df


def plot_coexpression_composition_from_inspection_table(
    inspection_df,
    dataset_order=None,
    sample_col="sample_label",
    coexpression_col="coexpression_cat",
    figsize=None,
    dpi=170,
    percent_text_threshold=1.0,
    cat_colors=None,
    labels_cond=None,
):
    """
    Plot observed coexpression composition directly from inspection_df.

    Produces two stacked-bar panels:
      - left:  LGR5+ crypts
      - right: LGR5- crypts

    Within each panel, bars are stacked over the 4 coexpression classes:
      0: Sero & NO Lyso
      1: Lyso & NO Sero
      2: NO (Sero | Lyso)
      3: Sero & Lyso

    Assumes coexpression_cat encoding:
      LGR5+ : 0,1,2,3
      LGR5- : 4,5,6,7
    """
    df = inspection_df.copy()

    if coexpression_col not in df.columns:
        raise KeyError(f"Missing required column: {coexpression_col}")

    df = df[df[coexpression_col].notna()].copy()
    if len(df) == 0:
        raise ValueError("No non-null coexpression_cat values found.")

    df[coexpression_col] = df[coexpression_col].astype(int)

    if dataset_order is None:
        dataset_order = list(pd.unique(df[sample_col]))

    if labels_cond is None:
        labels_cond = [
            "Sero & NO Lyso",
            "Lyso & NO Sero",
            "NO (Sero | Lyso)",
            "Sero & Lyso",
        ]

    if cat_colors is None:
        cat_colors = ["#d62728", "#9467bd", "#7f7f7f", "#ff7f0e"]

    n = len(dataset_order)

    # fixed bar width in data coordinates
    x = np.arange(n, dtype=float) * 0.78
    w = 0.30

    # figure width scales with number of datasets/timepoints
    if figsize is None:
        base_w = 5.0      # base width for two-panel frame
        per_bar_w = 0.75  # horizontal room per dataset
        fig_w = base_w + per_bar_w * n
        fig_h = 4.2 * 0.80
        figsize = (fig_w, fig_h)

    # -------------------------------------------------
    # build per-sample observed stacks
    # -------------------------------------------------
    stack_pos = []
    stack_neg = []
    Ns_pos = []
    Ns_neg = []

    for label in dataset_order:
        sub = df[df[sample_col] == label]
        cats = sub[coexpression_col].to_numpy(dtype=int)

        # LGR5+
        counts_pos = np.array([(cats == i).sum() for i in [0, 1, 2, 3]], dtype=float)
        n_pos = int(counts_pos.sum())
        frac_pos = counts_pos / n_pos if n_pos > 0 else np.full(4, np.nan)

        # LGR5-
        counts_neg = np.array([(cats == i).sum() for i in [4, 5, 6, 7]], dtype=float)
        n_neg = int(counts_neg.sum())
        frac_neg = counts_neg / n_neg if n_neg > 0 else np.full(4, np.nan)

        stack_pos.append(frac_pos)
        stack_neg.append(frac_neg)
        Ns_pos.append(n_pos)
        Ns_neg.append(n_neg)

    stack_pos = np.vstack(stack_pos)   # (Ndatasets, 4)
    stack_neg = np.vstack(stack_neg)   # (Ndatasets, 4)
    Ns_pos = np.asarray(Ns_pos, dtype=int)
    Ns_neg = np.asarray(Ns_neg, dtype=int)

    # -------------------------------------------------
    # plotting
    # -------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=figsize, dpi=dpi)
    fig.subplots_adjust(wspace=0.55, top=0.84)

    def stacked_single(ax, obs_2x4, Ns, ylabel):
        bottom = np.zeros(n, dtype=float)

        for ci in range(4):
            vals = 100.0 * obs_2x4[:, ci]

            bars = ax.bar(
                x,
                vals,
                width=w,
                bottom=bottom,
                color=cat_colors[ci],
                edgecolor="white",
                linewidth=0.7,
                zorder=3,
            )

            for rect, v, btm in zip(bars, vals, bottom):
                if not np.isfinite(v) or v < percent_text_threshold:
                    continue
                ax.text(
                    rect.get_x() + rect.get_width() / 2,
                    btm + v / 2,
                    f"{v:.0f}%",
                    ha="center",
                    va="center",
                    color="black",
                    fontsize=5,
                    fontweight="bold",
                    clip_on=True,
                    zorder=4,
                )

            bottom = bottom + np.nan_to_num(vals, nan=0.0)

        # N labels above bars
        trans = ax.get_xaxis_transform()
        for j, n_here in enumerate(Ns):
            ax.text(
                x[j], 1.01, f"N={int(n_here)}",
                transform=trans,
                ha="center",
                va="bottom",
                fontsize=5,
                fontweight="bold",
                clip_on=False,
            )

        ax.set_ylim(0, 100)
        ax.set_yticks([0, 25, 50, 75, 100])
        ax.grid(True, axis="y", linestyle="--", linewidth=0.5, alpha=0.5)

        ax.set_xticks(x)
        ax.set_xticklabels(dataset_order, fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.tick_params(axis="y", labelsize=7)

    stacked_single(axes[0], stack_pos, Ns_pos, "% LGR5+ crypts")
    stacked_single(axes[1], stack_neg, Ns_neg, "% LGR5- crypts")

    handles = [plt.Rectangle((0, 0), 1, 1, facecolor=cat_colors[i]) for i in range(4)]
    fig.legend(
        handles,
        labels_cond,
        frameon=False,
        ncol=2,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.02),
        fontsize=8,
    )

    export_figure(fig, "marker_coexpression_composition")
    plt.show()

    return {
        "stack_pos": stack_pos,
        "stack_neg": stack_neg,
        "Ns_pos": Ns_pos,
        "Ns_neg": Ns_neg,
        "dataset_order": dataset_order,
    }

plot_df = plot_lgr5_composition_from_inspection_table(
    inspection_df=inspection_df,
    dataset_order=SAMPLE_ORDER,
    sample_col="sample_label",
    lgr5_marker="LGR5",
)

lgr5_bi_category_df = build_lgr5_by_budding_index_category_table(
    inspection_df=inspection_df,
    group_order=SAMPLE_ORDER,
    lgr5_marker="LGR5",
)
plot_lgr5_bi_category_df = plot_lgr5_by_budding_index_category(
    lgr5_category_df=lgr5_bi_category_df,
    group_order=["all timepoints"] + SAMPLE_ORDER,
)

plot_coexpression_composition_from_inspection_table(
    inspection_df=inspection_df,
    dataset_order=SAMPLE_ORDER,
    sample_col="sample_label",
    coexpression_col="coexpression_cat",
)


## Export Tables

Export crypt-level and plotting summary tables using the same effective marker conventions as the figures.


In [ ]:
def _csv_safe_cell_ids(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    try:
        return ";".join(str(int(v)) for v in value)
    except Exception:
        return str(value)


def build_crypt_composition_export_table(inspection_df, markers=CRYPT_COMPOSITION_MARKERS):
    metadata_cols = [
        "dataset",
        "timepoint",
        "sample_label",
        "label_uid",
        "crypt_number",
        "crypt_number_filtered",
        "graph_path",
        "mesh_path",
        "seg_path",
        "has_seg",
        "num_cells",
        "graph_patch_size",
        "constriction",
        "is_budded",
        "crypt_shape_class",
        "elongation",
        "coexpression_cat",
        "crypt_length",
        "crypt_cmax",
        "crypt_slenderness",
        "crypt_fullness",
        "crypt_circ_pos_var",
        "crypt_circ_pos_spread",
        "crypt_circ_pos_spread_norm",
        "crypt_budding_index_sstar",
        "budding_index_category",
        "n_markers_present",
    ]
    out = inspection_df[[col for col in metadata_cols if col in inspection_df.columns]].copy()
    if "crypt_cells" in inspection_df.columns:
        out["crypt_cell_ids"] = inspection_df["crypt_cells"].apply(_csv_safe_cell_ids)

    for marker in markers:
        for suffix in ["n_pos", "frac_pos", "has"]:
            col = f"{marker}_{suffix}"
            if col in inspection_df.columns:
                out[col] = inspection_df[col]
            else:
                out[col] = np.nan

    out["marker_preprocessing_tag"] = marker_preprocess_cache_tag()
    out["marker_harmonization_enabled"] = bool(ENABLE_MARKER_HARMONIZATION)
    out["marker_exclusivity_enabled"] = bool(ENABLE_MARKER_EXCLUSIVITY)
    return out


def build_crypt_roi_composition_export_table(inspection_df, roi_frac, markers=CRYPT_COMPOSITION_MARKERS):
    if roi_frac is None:
        raise ValueError("roi_frac must not be None for ROI composition export")
    roi_frac = float(roi_frac)
    rows = []
    metadata_cols = [
        "dataset",
        "timepoint",
        "sample_label",
        "label_uid",
        "crypt_number",
        "crypt_number_filtered",
        "graph_path",
        "mesh_path",
        "seg_path",
        "has_seg",
        "num_cells",
        "graph_patch_size",
        "constriction",
        "is_budded",
        "crypt_shape_class",
        "elongation",
        "coexpression_cat",
        "crypt_length",
        "crypt_cmax",
        "crypt_slenderness",
        "crypt_fullness",
        "crypt_circ_pos_var",
        "crypt_circ_pos_spread",
        "crypt_circ_pos_spread_norm",
        "crypt_budding_index_sstar",
        "budding_index_category",
    ]
    for _, rec in inspection_df.iterrows():
        graph_path = rec.get("graph_path")
        seg_path = rec.get("seg_path")
        if graph_path is None or pd.isna(graph_path) or seg_path is None or pd.isna(seg_path):
            continue
        G = load_graph_cached(graph_path)
        seg = load_seg_cached(seg_path)
        if "d_crypts_graph" not in seg:
            raise KeyError(f"seg['d_crypts_graph'] missing for seg_path={seg_path}")
        roi_cells = get_roi_cells_for_crypt(
            crypt_cells=rec["crypt_cells"],
            dist_bottom=np.asarray(seg["d_crypts_graph"], dtype=float),
            crypt_number=int(rec["crypt_number"]),
            roi_frac=roi_frac,
        )
        markers_eff, marker_names = effective_marker_data_for_graph(G)
        X_roi = np.asarray(markers_eff[roi_cells], dtype=float) if roi_cells.size else np.zeros((0, len(marker_names)), dtype=float)
        row = {col: rec[col] for col in metadata_cols if col in inspection_df.columns}
        row["roi_frac"] = roi_frac
        row["roi_num_cells"] = int(roi_cells.size)
        row["roi_cell_ids"] = _csv_safe_cell_ids(roi_cells)
        for marker in markers:
            idx = resolve_marker_indices(marker_names, [marker])
            if idx and roi_cells.size > 0:
                n_pos = int(np.sum(X_roi[:, idx[0]] > 0))
                frac_pos = float(n_pos / roi_cells.size)
                has = int(n_pos >= MIN_POSITIVE)
            elif idx:
                n_pos = 0
                frac_pos = np.nan
                has = 0
            else:
                n_pos = np.nan
                frac_pos = np.nan
                has = np.nan
            row[f"{marker}_n_pos"] = n_pos
            row[f"{marker}_frac_pos"] = frac_pos
            row[f"{marker}_has"] = has
        row["roi_n_markers_present"] = sum(
            1 for marker in markers
            if pd.notna(row.get(f"{marker}_has")) and int(row.get(f"{marker}_has")) == 1
        )
        row["marker_preprocessing_tag"] = marker_preprocess_cache_tag()
        row["marker_harmonization_enabled"] = bool(ENABLE_MARKER_HARMONIZATION)
        row["marker_exclusivity_enabled"] = bool(ENABLE_MARKER_EXCLUSIVITY)
        rows.append(row)
    return pd.DataFrame(rows)


crypt_composition_export_df = build_crypt_composition_export_table(inspection_df)
crypt_roi_composition_export_df = build_crypt_roi_composition_export_table(
    inspection_df,
    roi_frac=CRYPT_COMPOSITION_ROI_FRAC,
)
crypt_composition_export_df.to_csv(os.path.join(EXPORT_DIR, "crypt_composition_by_crypt.csv"), index=False)
crypt_roi_composition_export_df.to_csv(
    os.path.join(EXPORT_DIR, f"crypt_composition_by_crypt_roi{str(CRYPT_COMPOSITION_ROI_FRAC).replace('.', 'p')}.csv"),
    index=False,
)

if "pooled_composition_df" in globals():
    pooled_composition_df.to_csv(os.path.join(EXPORT_DIR, "crypt_pooled_marker_composition.csv"), index=False)
if "pooled_composition_roi_df" in globals():
    pooled_composition_roi_df.to_csv(
        os.path.join(EXPORT_DIR, "crypt_pooled_marker_composition_bi_regions.csv"),
        index=False,
    )
if "marker_summary_df" in globals():
    marker_summary_df.to_csv(os.path.join(EXPORT_DIR, "crypt_marker_summary_by_sample.csv"), index=False)
if "plot_df" in globals() and isinstance(plot_df, pd.DataFrame):
    plot_df.to_csv(os.path.join(EXPORT_DIR, "lgr5_crypt_composition_summary.csv"), index=False)
if "lgr5_bi_category_df" in globals() and isinstance(lgr5_bi_category_df, pd.DataFrame):
    lgr5_bi_category_df.to_csv(os.path.join(EXPORT_DIR, "lgr5_crypt_composition_by_bi_category.csv"), index=False)
if "plot_lgr5_bi_category_df" in globals() and isinstance(plot_lgr5_bi_category_df, pd.DataFrame):
    plot_lgr5_bi_category_df.to_csv(os.path.join(EXPORT_DIR, "lgr5_crypt_composition_by_bi_category_plot.csv"), index=False)
if "plot_roi_df" in globals() and isinstance(plot_roi_df, pd.DataFrame):
    plot_roi_df.to_csv(
        os.path.join(EXPORT_DIR, "crypt_pooled_marker_composition_bi_regions_plot.csv"),
        index=False,
    )

print(f"Exported crypt composition tables to {EXPORT_DIR}")
crypt_composition_export_df.head()
